# Topic Modeling with LDA

## Introduction to Topic Modeling
Topic Modeling is an unsupervised machine learning technique used to discover hidden thematic structures in a large collection of documents. It helps in organizing, understanding, and summarizing large volumes of text.

## Latent Dirichlet Allocation (LDA)
Latent Dirichlet Allocation (LDA) is the most popular topic modeling algorithm. It treats each document as a mixture of topics and each topic as a mixture of words.

### Key Concepts:
1. **Document-Topic Distribution**: Probability distribution of topics in a specific document.
2. **Topic-Word Distribution**: Probability distribution of words associated with a specific topic.

### LDA Pipeline:
- **Input**: Raw text documents.
- **Data Preprocessing**: Lowercasing, removing punctuation, stop words, and lemmatization/stemming.
- **Vectorization**: Converting text into a numerical format, typically Term Frequency or Bag of Words (BoW) since LDA relies on term counts.
- **Model Training**: Fitting the LDA model to the Document-Term Matrix.
- **Topic Extraction**: Viewing the top words associated with each inferred topic.
- **Prediction**: Inferring the topic distribution of new documents.\n

In [26]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import re
import nltk
from nltk.corpus import stopwords

# Ensure stopwords are downloaded
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

## 1. Preparing the Data
We will create a small dataset of documents covering different themes (e.g., Technology, Sports, and Nature).\n

In [27]:
# Sample Data
documents = [
    "Apple is releasing a new smartphone with an advanced AI camera.",
    "The basketball team won the championship after a very close match.",
    "Global warming is affecting the natural habitats of polar bears in the Arctic.",
    "The new software update brings many cool features to the operating system.",
    "The football player scored three goals in the final tournament.",
    "Deforestation and pollution are major environmental concerns today.",
    "Tech giants are investing heavily in artificial intelligence research.",
    "The Olympic games will be held in Paris this year.",
    "Renewable energy sources are crucial for fighting climate change."
]

df = pd.DataFrame({'text': documents})

# Preprocess the text
df['clean_text'] = df['text'].apply(preprocess_text)
print("Cleaned Documents:")
print(df['clean_text'].head())

Cleaned Documents:
0    apple releasing new smartphone advanced ai camera
1             basketball team championship close match
2    global warming affecting natural habitats pola...
3    new software update brings many cool features ...
4    football player scored three goals final tourn...
Name: clean_text, dtype: object


## 2. Vectorization (Count Vectorizer)
LDA works best with raw word counts (Bag of Words) rather than TF-IDF. We use Scikit-Learn's `CountVectorizer` to create a document-term matrix.\n

In [32]:
# Initialize CountVectorizer
vectorizer = CountVectorizer(max_df=0.95, min_df=1, stop_words='english')

# Create Document-Term Matrix
dtm = vectorizer.fit_transform(df['clean_text'])

# Get feature names (words)
feature_names = vectorizer.get_feature_names_out()

print(f"Document-Term Matrix Shape: {dtm.shape}")
print(f"Number of Documents: {dtm.shape[0]}")
print(f"Number of Unique Words (Features): {dtm.shape[1]}")

Document-Term Matrix Shape: (9, 57)
Number of Documents: 9
Number of Unique Words (Features): 57


## 3. Training the LDA Model
We will train an LDA model to find `num_topics` topics within the documents.
Since our data naturally has 3 themes (Tech, Sports, Environment), we will set `n_components=3`.\n

In [37]:
# Define the LDA Model
num_topics = 4
lda_model = LatentDirichletAllocation(n_components=num_topics, random_state=42, max_iter=10)

# Fit the model to the document-term matrix
lda_model.fit(dtm)

print("LDA Model trained successfully.")

LDA Model trained successfully.


## 4. Extracting and Interpreting Topics
Let's look at the top words for each discovered topic.\n

In [38]:
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx + 1}:")
        # Get the indices of the top words
        top_indices = topic.argsort()[:-no_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]
        print(" ".join(top_words))
        print("-" * 20)

num_top_words = 4
display_topics(lda_model, feature_names, num_top_words)

Topic 1:
habitats polar affecting natural
--------------------
Topic 2:
match team basketball championship
--------------------
Topic 3:
new brings update operating
--------------------
Topic 4:
energy fighting crucial sources
--------------------


## 5. Topic Distribution for Documents
Now we can see how the LDA model assigns topic probabilities to our original documents.\n

In [39]:
# Get the topic distribution for the documents
topic_results = lda_model.transform(dtm)

# Assign the dominant topic to each document
df['Dominant_Topic'] = topic_results.argmax(axis=1) + 1

# Display the documents alongside their dominant topic
for index, row in df.iterrows():
    print(f"Doc {index + 1} | Topic {row['Dominant_Topic']} | {row['text'][:50]}...")

Doc 1 | Topic 4 | Apple is releasing a new smartphone with an advanc...
Doc 2 | Topic 2 | The basketball team won the championship after a v...
Doc 3 | Topic 1 | Global warming is affecting the natural habitats o...
Doc 4 | Topic 3 | The new software update brings many cool features ...
Doc 5 | Topic 3 | The football player scored three goals in the fina...
Doc 6 | Topic 3 | Deforestation and pollution are major environmenta...
Doc 7 | Topic 4 | Tech giants are investing heavily in artificial in...
Doc 8 | Topic 2 | The Olympic games will be held in Paris this year....
Doc 9 | Topic 4 | Renewable energy sources are crucial for fighting ...


## Conclusion
We successfully used Latent Dirichlet Allocation (LDA) to group text documents into different topics.\n